# Lab 32: Model Training Essentials: Datasets, Epochs, Loss and Evaluation

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
Until now you **used** models that other people trained. From this lab you **adapt** and later **build** models. Before you do, you need the words and the habits of training. In this lab you train a tiny model on a tiny dataset that you can see completely: sensor readings (temperature and humidity) and a label that says whether the fan should be on. You split the data into train, validation and test sets, train the model for many **epochs**, watch the **loss** go down, see **overfitting** happen on purpose, and evaluate the model honestly on data it has never seen. Everything runs in seconds, so you can try, break and fix. This lab is a **primer**: Part 4 of the course goes deeper into the same ideas.

**By the end of this lab you can:**
1. Explain a dataset, a feature, a label, and where labels come from.
2. Split data into train, validation and test sets and explain why each one exists.
3. Train a small neural network and read the loss and the accuracy over the epochs.
4. Recognise overfitting in the loss curves and name three ways to reduce it.
5. Evaluate a model on the test set with accuracy, a confusion matrix, precision and recall, and compare it with a baseline.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **Classification of this lab.** *Training from scratch* on a tiny example, to learn the vocabulary. In Lab 33 you fine-tune a vision model, in Lab 34 you train a wake-word model, and in Lab 35 you adapt a language model.

## 2. Background

**Training in one sentence.** A model has numbers inside (**weights**). Training changes the weights, a little at a time, so that the model's answers get closer to the correct answers in the **training data**.

| Word | Meaning |
|---|---|
| **Dataset** | A table of examples. Each row is one example. |
| **Feature** | A column the model sees as input (temperature, humidity) |
| **Label** | The correct answer for the example (fan on = 1, off = 0). Someone must decide it: a person, a rule, or a measurement. **Wrong labels teach wrong things.** |
| **Loss** | One number that says how wrong the model is. Training tries to make it smaller. |
| **Epoch** | One full pass over all the training examples |
| **Batch** | A small group of examples used for one update of the weights |
| **Learning rate** | How big each update is. Too big: the loss jumps. Too small: very slow. |

**Three data sets.**

| Set | Used for | Rule |
|---|---|---|
| **Train** | The model learns from it | The weights change here |
| **Validation** | You watch it during training and choose settings (epochs, size) | The weights do **not** change here, but your choices do |
| **Test** | One honest exam at the very end | Use it **once**. If you tune on it, it is no longer a test. |

**Overfitting.** A model that is too big, or trained too long on too little data, **memorises** the training examples, including their noise. The training loss keeps falling, but the validation loss starts to **rise**. The model is perfect on what it saw, and its loss on new data gets worse (it becomes over-confident, and on harder problems the accuracy falls too). Ways to reduce it: more data, a smaller model, stop earlier (**early stopping**), dropout.

**Evaluation.** Accuracy (the share of right answers) can lie. If 95 of 100 examples are `off`, a model that always says `off` has 95 % accuracy and is useless. So you also read the **confusion matrix** and:

| Metric | Question it answers |
|---|---|
| **Precision** | When the model says *fan on*, how often is it right? |
| **Recall** | Of all the cases where the fan *should* be on, how many did the model find? |
| **Baseline** | The score of the simplest possible answer (always the most common class). Your model must beat it. |

**Reproducible.** Use a fixed random seed, so you get the same numbers when you run the cell again.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(32)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (10 min)

### Step 1: Load the Tools
Run the cell. NumPy and pandas handle the data, scikit-learn splits it, PyTorch trains the model and matplotlib draws the pictures. The model is so small that it trains on the CPU in seconds.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print("numpy", np.__version__, "| torch", torch.__version__, "| seed", SEED)

---
# Part B: The Dataset (30 min)

### Step 1: Make the Data
The cell makes 400 sensor readings of a room. The **label** `fan` is 1 when the room is **too warm**: temperature above 27, or above 25 together with humidity above 70. Then 8 % of the labels are flipped on purpose, like the mistakes that real people make when they label. Look at the first rows.

In [ ]:
rng = np.random.default_rng(SEED)
N = 400
temperature = rng.uniform(18, 34, N).round(1)
humidity = rng.uniform(30, 90, N).round(0)
fan = ((temperature > 27) | ((temperature > 25) & (humidity > 70))).astype(int)
flip = rng.random(N) < 0.08                       # 8 % of the labels are wrong
fan = np.where(flip, 1 - fan, fan)

data = pd.DataFrame({"temperature": temperature, "humidity": humidity, "fan": fan})
print(data.head(8))
print()
print("Rows:", len(data), "| fan on:", int(data.fan.sum()), "| fan off:", int((1 - data.fan).sum()))
print("Wrong labels (flipped on purpose):", int(flip.sum()))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which columns are features and which is the label:
- What share of the examples have the fan on:
- Why can the model never be 100 % right on this data (hint: the wrong labels):

### Step 2: Look at the Data
Always **look** before you train. The picture shows every reading. Blue means fan off, red means fan on.

In [ ]:
plt.figure(figsize=(5, 4))
plt.scatter(data.temperature, data.humidity, c=data.fan, cmap="coolwarm", s=14)
plt.xlabel("temperature (C)")
plt.ylabel("humidity (%)")
plt.title("fan off (blue) and fan on (red)")
plt.show()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Where in the picture are the red points:
- Do you see red points inside the blue area? What are they:

---
# Part C: Split the Data (25 min)

### Step 1: Train, Validation and Test
`train_test_split` cuts the data at random. `stratify` keeps the share of fan on and off the same in every set. The test set is put aside and **not touched** until Part F.

In [ ]:
X = data[["temperature", "humidity"]].to_numpy(dtype=np.float32)
y = data["fan"].to_numpy(dtype=np.float32)

X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=SEED, stratify=y_rest)

for name, part in (("train", y_train), ("validation", y_val), ("test", y_test)):
    print(f"{name:<11} {len(part):>3} examples, fan on: {part.mean() * 100:.0f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The size of the three sets:
- Why is the share of fan on almost the same in each set:
- Why must the test set stay untouched until the end:

### Step 2: Scale the Features
Temperature is around 25 and humidity around 60. A network learns better when all inputs are around 0 with a spread of about 1. **Fit the scaler on the training set only** (the mean and the spread come from the train set) and then use the same numbers for the other sets. If you used the test set to calculate them, information from the test would leak into training.

In [ ]:
mean, spread = X_train.mean(axis=0), X_train.std(axis=0)


def scale(x):
    return (x - mean) / spread


Xt, Xv, Xs = (torch.tensor(scale(a)) for a in (X_train, X_val, X_test))
yt, yv, ys = (torch.tensor(b).unsqueeze(1) for b in (y_train, y_val, y_test))
print("mean of the train features:", mean.round(1), "| spread:", spread.round(1))
print("train features after scaling: mean", Xt.mean(0).numpy().round(2), "spread", Xt.std(0).numpy().round(2))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The mean and spread you got:
- What would be wrong with fitting the scaler on all the data:

---
# Part D: Train a Tiny Network (40 min)

### Step 1: The Model and the Training Loop
The network has two inputs, one hidden layer of 8 neurons and one output (a number between 0 and 1: how sure the model is that the fan must be on). The **loss** is binary cross entropy. **Adam** changes the weights. Each epoch runs over the training set in batches of 32, then measures the loss and the accuracy on the training and the validation set. The weights change **only** on the training batches.

In [ ]:
def make_model(hidden=8):
    return nn.Sequential(nn.Linear(2, hidden), nn.ReLU(), nn.Linear(hidden, 1), nn.Sigmoid())


def accuracy(model, X, y):
    with torch.no_grad():
        return ((model(X) > 0.5).float() == y).float().mean().item()


def train(model, X, y, epochs=60, lr=0.02, batch=32, X_val=None, y_val=None):
    """Train the model. Returns the history: loss and accuracy of the training set and of the validation set for each epoch."""
    loss_fn = nn.BCELoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"loss": [], "val_loss": [], "acc": [], "val_acc": []}
    for epoch in range(epochs):
        order = torch.randperm(len(X))
        for start in range(0, len(X), batch):
            idx = order[start:start + batch]
            optimiser.zero_grad()
            loss_fn(model(X[idx]), y[idx]).backward()
            optimiser.step()
        with torch.no_grad():
            history["loss"].append(loss_fn(model(X), y).item())
            history["acc"].append(accuracy(model, X, y))
            if X_val is not None:
                history["val_loss"].append(loss_fn(model(X_val), y_val).item())
                history["val_acc"].append(accuracy(model, X_val, y_val))
    return history


torch.manual_seed(SEED)
model = make_model()
history = train(model, Xt, yt, epochs=60, X_val=Xv, y_val=yv)
for epoch in (0, 4, 9, 19, 39, 59):
    print(f"epoch {epoch + 1:>2}: loss {history['loss'][epoch]:.3f}  val loss {history['val_loss'][epoch]:.3f}  accuracy {history['acc'][epoch] * 100:.0f} %  val accuracy {history['val_acc'][epoch] * 100:.0f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Loss and validation accuracy at the first and the last epoch printed:
- Number of weights in the model (count: 2x8 + 8 + 8x1 + 1):
- Why does the loss not reach 0:

### Step 2: The Learning Curves
The curves of the loss and of the accuracy tell the story of the training. A healthy run: both losses go down and stay close to each other.

In [ ]:
def plot_history(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
    axes[0].plot(history["loss"], label="train")
    axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_title("loss " + title)
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[1].plot(history["acc"], label="train")
    axes[1].plot(history["val_acc"], label="validation")
    axes[1].set_title("accuracy " + title)
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    plt.show()


plot_history(history)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Describe the curve of the loss in one sentence:
- Are the train and the validation curves close to each other? What does it mean:

### Step 3: Change One Thing: the Learning Rate
Train three models that differ only in the learning rate. Compare the loss after 30 epochs.

In [ ]:
for lr in (0.0005, 0.02, 1.0):
    torch.manual_seed(SEED)
    m = make_model()
    h = train(m, Xt, yt, epochs=30, lr=lr, X_val=Xv, y_val=yv)
    print(f"learning rate {lr:<7} loss after 30 epochs: {h['loss'][-1]:.3f}   validation accuracy: {h['val_acc'][-1] * 100:.0f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which learning rate was too small, which was good, which was too big:
- What did the too-big learning rate do to the loss:

---
# Part E: Overfitting on Purpose (30 min)

### Step 1: Too Little Data, Too Big a Model
The cell trains a big network (200 hidden neurons) on only 24 training examples for 400 epochs. Watch the two losses.

In [ ]:
torch.manual_seed(SEED)
small_X, small_y = Xt[:24], yt[:24]
big = make_model(hidden=200)
history_big = train(big, small_X, small_y, epochs=400, lr=0.01, batch=8, X_val=Xv, y_val=yv)
print(f"train loss {history_big['loss'][-1]:.3f}  train accuracy {history_big['acc'][-1] * 100:.0f} %")
print(f"validation loss {history_big['val_loss'][-1]:.3f}  validation accuracy {history_big['val_acc'][-1] * 100:.0f} %")
print("Lowest validation loss was at epoch", int(np.argmin(history_big["val_loss"])) + 1, "of 400")
plot_history(history_big, "(overfitting)")

> **Look at the numbers carefully.** The validation **accuracy** can still be quite high, because this problem is easy (only two features). The damage shows in the **validation loss**: it rose from about 0.3 to above 3. The model became **over-confident**: it is very sure, and when it is wrong it is wrong by a lot. On a harder problem the accuracy would fall as well.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Train accuracy and validation accuracy at the end:
- At which epoch was the validation loss lowest, and what happened to the loss after it:
- Say in your own words why the training loss reaches 0 but the validation loss rises:

### Step 2: Reduce the Overfitting
Three fixes, one at a time. The cell trains again and prints the validation loss and accuracy of each. **Early stopping** keeps the weights of the epoch with the lowest validation loss.

In [ ]:
def run(label, hidden, X, y, epochs, early=False):
    torch.manual_seed(SEED)
    m = make_model(hidden)
    best = (9.0, None)
    h = {"loss": [], "val_loss": [], "acc": [], "val_acc": []}
    for epoch in range(epochs):
        part = train(m, X, y, epochs=1, lr=0.01, batch=8, X_val=Xv, y_val=yv)
        for key in h:
            h[key] += part[key]
        if early and h["val_loss"][-1] < best[0]:
            best = (h["val_loss"][-1], {k: v.clone() for k, v in m.state_dict().items()})
    if early:
        m.load_state_dict(best[1])
    with torch.no_grad():
        final_loss = nn.BCELoss()(m(Xv), yv).item()
    print(f"{label:<38} validation accuracy {accuracy(m, Xv, yv) * 100:.0f} %   validation loss at the end {final_loss:.3f}")
    return m


run("A: big model, 24 examples, 400 epochs", 200, small_X, small_y, 400)
run("B: small model (4 neurons), 24 examples", 4, small_X, small_y, 400)
run("C: big model, all 240 train examples", 200, Xt, yt, 100)
run("D: big model, 24 examples, early stop", 200, small_X, small_y, 400, early=True)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The four validation losses at the end (A, B, C, D):
- Which fix helped the most, and which would you use in a real project:
- What does early stopping use to decide when to stop:

---
# Part F: The Honest Exam (30 min)

### Step 1: The Baseline and the Test Set
The model of Part D is evaluated **once** on the test set. First the baseline: always answer with the most common class. Then accuracy, the confusion matrix, precision and recall.

In [ ]:
with torch.no_grad():
    predicted = (model(Xs) > 0.5).float().squeeze().numpy()
truth = ys.squeeze().numpy()

majority = float(truth.mean() >= 0.5)
baseline = (truth == majority).mean()
tp = int(((predicted == 1) & (truth == 1)).sum())
tn = int(((predicted == 0) & (truth == 0)).sum())
fp = int(((predicted == 1) & (truth == 0)).sum())
fn = int(((predicted == 0) & (truth == 1)).sum())
print(f"Baseline (always {'on' if majority else 'off'}): accuracy {baseline * 100:.0f} %")
print(f"Model: accuracy {(tp + tn) / len(truth) * 100:.0f} %")
print()
print("                 model says off   model says on")
print(f"truth: off       {tn:>10}   {fp:>14}")
print(f"truth: on        {fn:>10}   {tp:>14}")
print()
print(f"precision {tp / max(1, tp + fp) * 100:.0f} %   recall {tp / max(1, tp + fn) * 100:.0f} %")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Baseline accuracy and model accuracy:
- True positives, false positives, false negatives, true negatives:
- Which is worse for a fan controller: a false negative or a false positive? Explain:

### Step 2: The Accuracy Trap
Make a dataset where only 5 % of the examples are *fan on*. A model that always says *off* gets 95 % and finds nothing. Look at its recall.

In [ ]:
truth_rare = np.array([1] * 5 + [0] * 95)
always_off = np.zeros(100)
print("accuracy of 'always off':", (always_off == truth_rare).mean() * 100, "%")
print("recall of 'always off':  ", ((always_off == 1) & (truth_rare == 1)).sum() / truth_rare.sum() * 100, "%")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Accuracy and recall of the model that always says off:
- Name a real task where this trap is dangerous:

---
# Part G: Make It Your Own and Present (25 min)
Choose **one** and write down what you find:

- Change the rule that makes the labels (for example a different temperature limit) and train again.
- Add a third feature (for example *hour of the day*) to the data and the model.
- Raise the number of wrong labels from 8 % to 25 %. What happens to the accuracy limit and to the curves?
- Train the same model 5 times with different seeds and write the five test accuracies. How much do they differ?

Then give a **3-minute demonstration**: show the data, the split, the learning curves, the overfitting run and the confusion matrix.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- What you saw:
- Two things you will do differently in your next training project:

> **Where next.** Now you can read the words of training. Lab 33 fine-tunes a YOLO detector on your own pictures, Lab 34 trains a wake word, and Lab 35 adapts a language model with LoRA. In Part 4 you build the same ideas yourself, from the data to the model.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The data is made, looked at, and the features and the label are named | |
| 2 | The train, validation and test sets are made, and the scaler is fitted on the train set only | |
| 3 | The network is trained and the learning curves are read | |
| 4 | Three learning rates are compared and the result is explained | |
| 5 | The overfitting run shows the validation loss rising, and the four fixes are compared | |
| 6 | The test set is used once: baseline, confusion matrix, precision and recall | |
| 7 | The accuracy trap is explained | |
| 8 | One improvement of Part G is done and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What is the difference between a feature and a label? Where can a wrong label come from?:
- 2. Why do we need three sets and not only two?:
- 3. What is an epoch? What is a batch?:
- 4. The training loss falls and the validation loss rises. What is happening? Name two fixes.:
- 5. Why must the scaler be fitted on the training set only?:
- 6. What does the learning rate do? What do you see when it is far too big?:
- 7. A model has 98 % accuracy but the baseline has 97 %. Is the model good? Explain.:
- 8. Explain precision and recall in one sentence each.:
- 9. Why is the test set used only once?:
- 10. A fan controller must never miss a very hot room. Which metric do you watch, and what do you do about it?:

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `No module named 'sklearn'` or `'matplotlib'` | The library is not in your environment | Call your instructor. In Lab 19 the first one was installed; do not install other things on your own. |
| The pictures do not appear | The plot cell was not run or the plot window is closed | Run the cell again; the picture appears under it |
| The numbers are different from your classmate's | The seed or the order of the cells is different | Run the cells from the top. The seed is 42. |
| The loss is `nan` | The learning rate is far too big | Use 0.02 or smaller |
| The validation accuracy is higher than the training accuracy | The validation set is small, so the numbers jump | This can happen with 80 examples. It is not a bug. |
| `RuntimeError: mat1 and mat2 shapes cannot be multiplied` | You changed the number of features but not the first layer | Change `nn.Linear(2, ...)` to the new number of features |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(32)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- scikit-learn, train_test_split: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- PyTorch, training a classifier: https://pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html
- Google, Machine Learning Crash Course (overfitting, evaluation): https://developers.google.com/machine-learning/crash-course